# VisionOps HAR Research — Full End-to-End Run

**All improvements active by default:**
- ✅ All available clips (no 100-clip cap)
- ✅ 3-view InHARD extraction (top-down quadrant by default)
- ✅ Temporal attention pooling (replaces mean-pool)
- ✅ Focal Loss (γ=2) + WeightedRandomSampler
- ✅ Mixup augmentation in embedding space
- ✅ 100 epochs + CosineAnnealingLR
- ✅ Optional SupCon pre-training stage
- ✅ Optional Bi-GRU sequential head
- ✅ Full analysis: UMAP clusters, frame strips, YOLO detection, per-person heatmaps

In [1]:
import sys, os
from pathlib import Path

# Add har-research to path so `lib` imports work
NB_DIR = Path("__file__").parent.resolve() if "__file__" in dir() else Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

print(f"Working dir : {NB_DIR}")
print(f"Python      : {sys.version.split()[0]}")

Working dir : /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/har-research
Python      : 3.12.11


## ⚙️ Configuration
Edit these before running. Sensible defaults are pre-set.

In [2]:
from lib.pipeline import PipelineConfig
from lib.constants import BACKBONE_VJEPA, BACKBONE_DINOV2

cfg = PipelineConfig(
    # ── Data ────────────────────────────────────────────────────────
    clips_per_class  = None,          # None = use ALL available clips
    inhard_view      = "topdown",     # "topdown" | "side" | "front" | "full"

    # ── Backbone(s) to run ────────────────────────────────────────────
    backbones        = (BACKBONE_VJEPA, BACKBONE_DINOV2),
    temporal_agg     = "attention",   # "attention" | "mean"

    # ── Classifier head ──────────────────────────────────────────────
    head_arch        = "mlp",         # "mlp" | "gru"
    train_epochs     = 100,

    # ── Loss & sampling improvements ─────────────────────────────────
    use_focal_loss       = True,
    focal_gamma          = 2.0,
    use_weighted_sampler = True,
    use_mixup            = True,
    mixup_alpha          = 0.2,
    mixup_n_aug          = 2,

    # ── SupCon (optional, adds ~50 extra epochs before classification) ─
    use_supcon       = False,         # set True for best results
    supcon_epochs    = 50,

    # ── Evaluation split ─────────────────────────────────────────────
    split_mode       = "subject",     # "subject" (honest) | "random"
    analysis_split   = "subject",

    # ── Pipeline gates ────────────────────────────────────────────────
    run_data_check   = True,
    run_embeddings   = True,
    run_train        = True,
    run_analysis     = True,
    run_compare      = True,
    run_eval_video   = False,         # set True to render annotated mock video

    # ── Cache / skip ─────────────────────────────────────────────────
    skip_embeddings_if_exists = False,
    skip_train_if_exists      = False,
)

print("Config ready:")
print(f"  view={cfg.inhard_view}  temporal_agg={cfg.temporal_agg}  head={cfg.head_arch}")
print(f"  epochs={cfg.train_epochs}  focal={cfg.use_focal_loss}  mixup={cfg.use_mixup}  supcon={cfg.use_supcon}")
print(f"  backbones={cfg.backbones}")

Config ready:
  view=topdown  temporal_agg=attention  head=mlp
  epochs=100  focal=True  mixup=True  supcon=False
  backbones=('vjepa', 'dinov2')


## 🚀 Run Full Pipeline

In [3]:
from lib.pipeline import run_pipeline

results = run_pipeline(cfg)
print(f"\nPipeline status: {results['status']}")

[01] Clips: 3425 across 12 classes
[01]   view=topdown  clips_per_class=ALL
[01]   imbalance ratio=16.4x  (max=641 min=39)
[01] Mock videos: ['Industrial-One.mp4', 'madera.mp4']
[02] Extracting embeddings for 3425 clips · view=topdown
[02] Pass 1: YOLO crops …


YOLO crops:   0%|          | 0/3425 [00:00<?, ?it/s]

[02]   3425/3425 clips had valid crops
[02] Pass 2: vjepa encoding 3425 clips …


vjepa encode:   0%|          | 0/3425 [00:00<?, ?it/s]

[02]   Saved (3423, 1024) → embeddings.npz
[02]   Class distribution: {'Consult sheets': 132, 'Picking in front': 456, 'Picking left': 641, 'Put down component': 385, 'Put down measuring rod': 74, 'Put down screwdriver': 416, 'Put down subsystem': 77, 'Take component': 485, 'Take measuring rod': 76, 'Take screwdriver': 420, 'Take subsystem': 39, 'Turn sheets': 224}
[02] Pass 2: dinov2 encoding 3425 clips …


...

Using cache found in /Users/cpanoh/.cache/torch/hub/facebookresearch_dinov2_main
/Users/cpanoh/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/cpanoh/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/cpanoh/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


[02]   Saved (3425, 1024) → embeddings_dinov2.npz
[02]   Class distribution: {'Consult sheets': 132, 'Picking in front': 456, 'Picking left': 641, 'Put down component': 385, 'Put down measuring rod': 74, 'Put down screwdriver': 416, 'Put down subsystem': 77, 'Take component': 485, 'Take measuring rod': 76, 'Take screwdriver': 420, 'Take subsystem': 39, 'Turn sheets': 224}
[03] Training vjepa head …
Mixup: 2922 → 8766 training samples
FocalLoss(gamma=2.0) + class weights
HarMLP  emb_dim=1024  n_classes=12

Training MLP head | 100 epochs | device=cpu
  train=8766 (+aug) val=501 classes=12
  ep   1/100  train=2.5812  val=1.5615  val_acc=25.5%  lr=1.00e-03
  ep  10/100  train=0.7510  val=1.5459  val_acc=44.1%  lr=9.76e-04
  ep  20/100  train=0.5390  val=2.1101  val_acc=50.5%  lr=9.05e-04
  ep  30/100  train=0.4551  val=2.2840  val_acc=51.9%  lr=7.96e-04
  ep  40/100  train=0.3818  val=2.4310  val_acc=55.5%  lr=6.58e-04
  ep  50/100  train=0.3404  val=2.6210  val_acc=56.3%  lr=5.05e-04
  ep

/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/har-research/lib/har_analysis.py:230: UserWarning: umap-learn not installed; falling back to t-SNE
  warnings.warn("umap-learn not installed; falling back to t-SNE")


[06]   Analysis failed (vjepa): TSNE.__init__() got an unexpected keyword argument 'n_iter'
[06] Analysis dinov2 …
[06]   Analysis failed (dinov2): TSNE.__init__() got an unexpected keyword argument 'n_iter'
[compare] Winner: dinov2  F1=0.4254  acc=0.4152
[pipeline] Summary → /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/har-research/outputs/pipeline_v2_run_summary.json

Pipeline status: ok


/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/har-research/lib/har_analysis.py:230: UserWarning: umap-learn not installed; falling back to t-SNE
  warnings.warn("umap-learn not installed; falling back to t-SNE")


## 📊 Results Summary

In [4]:
import json
from lib.paths import OUTPUTS_DIR

# Print per-backbone metrics
train_step = results.get("steps", {}).get("03_train", {})
for backbone, stats in train_step.items():
    if stats.get("skipped"):
        print(f"{backbone}: skipped")
        continue
    r = stats.get("val_report", {})
    acc = r.get("accuracy", "?")
    mf  = r.get("macro avg", {}).get("f1-score", "?")
    print(f"\n{'='*50}")
    print(f"  backbone   : {backbone}")
    if isinstance(acc, float): print(f"  accuracy   : {acc:.1%}")
    if isinstance(mf,  float): print(f"  macro F1   : {mf:.3f}")
    si = stats.get("split_info", {})
    print(f"  head       : {si.get('head_arch','?')}")
    print(f"  train/val  : {si.get('n_train_orig','?')} / {si.get('n_val','?')}")
    print(f"  aug train  : {si.get('n_train_aug','?')} (after mixup)")
    if isinstance(r, dict):
        print(f"\n  Per-class F1:")
        for cls, v in r.items():
            if isinstance(v, dict) and "f1-score" in v:
                bar = '█' * int(v['f1-score'] * 20)
                print(f"    {cls:30s} {v['f1-score']:.2f} {bar}")


  backbone   : vjepa
  accuracy   : 32.7%
  macro F1   : 0.321
  head       : mlp
  train/val  : 2922 / 501
  aug train  : 8766 (after mixup)

  Per-class F1:
    Consult sheets                 0.29 █████
    Picking in front               0.48 █████████
    Picking left                   0.12 ██
    Put down component             0.19 ███
    Put down measuring rod         0.20 ███
    Put down screwdriver           0.39 ███████
    Put down subsystem             0.44 ████████
    Take component                 0.14 ██
    Take measuring rod             0.14 ██
    Take screwdriver               0.57 ███████████
    Take subsystem                 0.38 ███████
    Turn sheets                    0.52 ██████████
    macro avg                      0.32 ██████
    weighted avg                   0.33 ██████

  backbone   : dinov2
  accuracy   : 41.5%
  macro F1   : 0.425
  head       : mlp
  train/val  : 2924 / 501
  aug train  : 8772 (after mixup)

  Per-class F1:
    Consult sheets      

## 📈 Analysis Charts
Charts are saved to `outputs/har_analysis_v2/<date>_<tag>/`. Open the REPORT.md for links.

In [5]:
from IPython.display import Image, display, Markdown

analysis_step = results.get("steps", {}).get("06_analysis", {})
for backbone, info in analysis_step.items():
    if info.get("skipped") or info.get("error"):
        continue
    print(f"\n=== {backbone} ===  acc={info.get('accuracy',0):.1%}  macro_f1={info.get('macro_f1',0):.3f}")
    charts = info.get("charts", {})
    # Display key charts inline
    for key in ["02_confusion_matrix", "05_umap_clusters", "08_per_person_heatmap", "09_confidence_calibration"]:
        path = charts.get(key.split('_',1)[1].replace('_',' ').strip()) or ""
        # find by value
        for k, v in charts.items():
            if v and key.split('_',1)[-1] in (v or ''):
                path = v
                break
        if path and Path(path).is_file():
            display(Markdown(f"**{Path(path).stem}**"))
            display(Image(path, width=700))

## 🏆 Backbone Comparison

In [6]:
import pandas as pd

compare = results.get("steps", {}).get("07_compare", {})
if compare and "results" in compare:
    df = pd.DataFrame(compare["results"])
    display(df[[c for c in ["backbone","accuracy","macro_f1","weighted_f1","n_test","status"] if c in df.columns]])
    if "winner" in compare:
        w = compare["winner"]
        print(f"\n🏆 Winner: {w['backbone']}  macro_F1={w['macro_f1']:.4f}  accuracy={w['accuracy']:.4f}")
else:
    print("Compare step not run or only one backbone.")

,backbone,accuracy,macro_f1,weighted_f1,n_test,status
0,vjepa,0.327345,0.320652,0.330324,501,ok
1,dinov2,0.415170,0.425447,0.417590,501,ok



🏆 Winner: dinov2  macro_F1=0.4254  accuracy=0.4152
